# Agent Tool-Calling Loop — Single-Notebook Demo (Real LLM)

**DevRev Technical Round, Section 3.** A ReAct-style tool-calling agent, built as
**one self-contained notebook** rather than spread across a `src/` package. The
"brain" that decides which tool to call is a **real LLM** (OpenAI, via native
function/tool calling) — not a rule-based stand-in.

What this notebook covers, end to end:

- A **tool registry**: name, description, JSON-schema args, a `destructive` flag
- The **ReAct loop** itself: think → act → observe → repeat, with a **max-iteration guard**
- Routing a query to the right tool using an **LLM's native tool calling**
- **Synthesizing a final answer** once the LLM stops requesting tools
- A **confirmation gate** before destructive actions (close/delete)
- **Memoization** of repeated tool calls within a session
- **Retry** on transient failures, and **fallback** to a backup tool on permanent ones
- **Disambiguation** when two tools could plausibly answer the same query
- **State across conversation turns** — a `Session` holding message history + a memo cache
- **Parallel vs. sequential tool calls**, and why it matters for latency
- **Observability** — a full trace of every tool call, its status, and its latency

Run the cells top to bottom. You'll be prompted for an `OPENAI_API_KEY` the first time
one is needed (only Sections 3 onward actually call the API — everything before that
is plain Python you can run and inspect for free).

## 0. Setup

In [ ]:
# pip install openai   # the only external dependency this notebook needs

import os, json, time, getpass
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4o-mini"   # cheap + fast + supports tool calling; swap for gpt-4o if you like

## 1. Tools — the atomic actions the agent can call

A tool is: a **name**, a **description** (what the LLM reads to decide when to use it),
a JSON-schema for its **arguments**, an **implementation**, and whether it's
**destructive** (close/delete → needs a confirmation gate before it runs).

The backend here is an in-memory stand-in for the DevRev ticket API — enough to make
the tool calls do something real without needing live credentials.

In [ ]:
class TransientError(Exception):
    """A temporary failure (network blip, 503) -- safe to RETRY."""

class NotFoundError(Exception):
    """A permanent 'no such record' failure -- do NOT retry; consider a fallback."""


class TicketStore:
    """Stands in for the DevRev API -- just a dict of tickets in memory."""

    def __init__(self):
        self.tickets = {
            "TKT-1": {"id": "TKT-1", "subject": "Cannot log in", "status": "open", "tags": ["auth"]},
            "TKT-2": {"id": "TKT-2", "subject": "Billing overcharge", "status": "open", "tags": ["billing"]},
            "TKT-3": {"id": "TKT-3", "subject": "Login page 500 error", "status": "open", "tags": ["auth", "bug"]},
        }
        self._next = 4

    def search(self, query):
        q = (query or "").lower()
        hits = [t for t in self.tickets.values()
                if q in t["subject"].lower() or any(q in tag for tag in t["tags"])]
        return [{"id": t["id"], "subject": t["subject"], "status": t["status"]} for t in hits]

    def get(self, ticket_id):
        if ticket_id not in self.tickets:
            raise NotFoundError(f"ticket {ticket_id!r} does not exist")
        return dict(self.tickets[ticket_id])

    def create(self, subject):
        tid = f"TKT-{self._next}"; self._next += 1
        self.tickets[tid] = {"id": tid, "subject": subject, "status": "open", "tags": []}
        return dict(self.tickets[tid])

    def close(self, ticket_id):                 # DESTRUCTIVE
        if ticket_id not in self.tickets:
            raise NotFoundError(f"ticket {ticket_id!r} does not exist")
        self.tickets[ticket_id]["status"] = "closed"
        return dict(self.tickets[ticket_id])


store = TicketStore()

# name -> python implementation
TOOL_IMPLS = {
    "search_tickets": lambda query: store.search(query),
    "get_ticket": lambda ticket_id: store.get(ticket_id),
    "create_ticket": lambda subject: store.create(subject),
    "close_ticket": lambda ticket_id: store.close(ticket_id),
}
DESTRUCTIVE_TOOLS = {"close_ticket"}

# what we hand the LLM so it knows which tools exist and how to call them
TOOLS_SPEC = [
    {"type": "function", "function": {
        "name": "search_tickets",
        "description": "Find open tickets whose subject or tags match a query string.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "keyword to search for"}},
            "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "get_ticket",
        "description": "Fetch full details of one ticket by its id, e.g. 'TKT-1'.",
        "parameters": {"type": "object", "properties": {
            "ticket_id": {"type": "string"}}, "required": ["ticket_id"]},
    }},
    {"type": "function", "function": {
        "name": "create_ticket",
        "description": "Create a new ticket with a subject.",
        "parameters": {"type": "object", "properties": {
            "subject": {"type": "string"}}, "required": ["subject"]},
    }},
    {"type": "function", "function": {
        "name": "close_ticket",
        "description": "Close (resolve) a ticket by id. DESTRUCTIVE: requires confirmation.",
        "parameters": {"type": "object", "properties": {
            "ticket_id": {"type": "string"}}, "required": ["ticket_id"]},
    }},
]

## 2. Robustness — confirmation gate, memoization, retry, fallback, disambiguation

One function, `execute_tool`, composes the first three — they apply to *every* call.
Fallback and disambiguation are separate helpers you reach for explicitly, since they
need a second tool/candidate in hand.

In [ ]:
class ConfirmationRequired(Exception):
    """Raised when a destructive tool is called without approval.

    Note: the payload is stored on `call_args`, NOT `args` -- `BaseException.args`
    is a reserved attribute (a tuple used to build the default message). Overwriting
    it with a dict silently breaks `str(exc)`. Easy mistake, worth knowing about.
    """
    def __init__(self, tool, args):
        super().__init__(f"'{tool}' needs confirmation for {args}")
        self.tool, self.call_args = tool, args


def deny_destructive(tool, args):     # the SAFE default confirmation policy
    return False

def always_approve(tool, args):       # for demos / trusted flows
    return True


def _cache_key(tool_name, args):
    return tool_name + "::" + json.dumps(args, sort_keys=True, default=str)


def execute_tool(tool_name, args, *, log, impl=None, memo=None, confirm=deny_destructive, retries=3):
    """Run one tool call safely: confirmation gate -> memo cache -> retry."""
    impl = impl if impl is not None else TOOL_IMPLS.get(tool_name)
    if impl is None:
        raise ValueError(f"no implementation registered for tool {tool_name!r}")

    if tool_name in DESTRUCTIVE_TOOLS and not confirm(tool_name, args):
        log.append({"tool": tool_name, "args": args, "status": "blocked", "note": "needs confirmation"})
        raise ConfirmationRequired(tool_name, args)

    key = _cache_key(tool_name, args)
    if memo is not None and key in memo:
        log.append({"tool": tool_name, "args": args, "status": "cache_hit", "result": memo[key]})
        return memo[key]

    attempt = 0
    while True:
        t0 = time.perf_counter()
        try:
            result = impl(**args)
            ms = round((time.perf_counter() - t0) * 1000, 1)
            log.append({"tool": tool_name, "args": args, "status": "ok", "result": result, "ms": ms})
            if memo is not None:
                memo[key] = result
            return result
        except TransientError as exc:
            if attempt < retries:
                attempt += 1
                log.append({"tool": tool_name, "args": args, "status": "retry", "note": str(exc)})
                time.sleep(0.05 * attempt)          # tiny backoff (real code: + jitter)
                continue
            log.append({"tool": tool_name, "args": args, "status": "error", "note": str(exc)})
            raise
        except Exception as exc:
            log.append({"tool": tool_name, "args": args, "status": "error", "note": str(exc)})
            raise


def with_fallback(tool_name, fallback_name, args, *, log, impl=None, fallback_impl=None, **kw):
    """Try `tool_name`; on a permanent failure, run `fallback_name` instead."""
    try:
        return execute_tool(tool_name, args, log=log, impl=impl, **kw)
    except ConfirmationRequired:
        raise                                        # confirmation isn't a failure to fall back on
    except Exception:
        log.append({"tool": tool_name, "args": args, "status": "error",
                     "note": f"falling back to {fallback_name}"})
        return execute_tool(fallback_name, args, log=log, impl=fallback_impl, **kw)


def disambiguate(candidate_names, query):
    """Deterministic tie-break when two tools could both plausibly answer a query.

    Prefers the non-destructive tool, then the one whose description shares the most
    words with the query. A real LLM usually disambiguates itself via good tool
    descriptions -- keep this as the safety net for when it's genuinely unsure.
    """
    words = set(query.lower().split())
    specs = {t["function"]["name"]: t["function"]["description"] for t in TOOLS_SPEC}

    def score(name):
        overlap = len(words & set(specs[name].lower().split()))
        return (0 if name in DESTRUCTIVE_TOOLS else 1, overlap)

    return max(candidate_names, key=score)


def make_flaky(func, fail_times=2):
    """Wrap a function so it raises TransientError on its first `fail_times` calls."""
    state = {"n": 0}
    def wrapper(**kwargs):
        state["n"] += 1
        if state["n"] <= fail_times:
            raise TransientError(f"temporary outage (attempt {state['n']})")
        return func(**kwargs)
    return wrapper


def print_trace(log):
    """Human-readable table of every tool call, in order -- the observability trace."""
    if not log:
        print("(no tool calls)"); return
    print(f"{'#':>2}  {'tool':<16} {'status':<10} {'ms':>7}  args -> result/note")
    print("-" * 90)
    for i, r in enumerate(log, 1):
        outcome = r.get("note", r.get("result"))
        ms = r.get("ms", "")
        print(f"{i:>2}  {r['tool']:<16} {r['status']:<10} {str(ms):>7}  {r['args']} -> {outcome}")

## 3. The brain — a real LLM decides the next action

No rule-based router, no string matching: the model sees the tool schemas plus the
conversation so far, and either **(a)** requests one or more tool calls, or **(b)**
answers in plain text. That's OpenAI's native tool-calling contract, and it's exactly
the shape a production "brain" (LangChain's `bind_tools`, the raw Responses API, etc.)
would use.

In [ ]:
SYSTEM_PROMPT = (
    "You are a DevRev-style support agent. Use the available tools to satisfy the "
    "user's request -- don't guess ticket ids or content you haven't looked up. "
    "Call tools one step at a time, unless two calls are genuinely independent of "
    "each other, in which case you may request them together in the same turn. "
    "Once you have everything you need, reply with a final, plain-text answer and "
    "no further tool calls."
)

def llm_step(messages, tools=TOOLS_SPEC, model=MODEL):
    """One call to the LLM: returns its next message (tool_calls, or a final answer)."""
    resp = client.chat.completions.create(model=model, messages=messages, tools=tools, tool_choice="auto")
    return resp.choices[0].message

## 4. The ReAct loop — think, act, observe, repeat

`Session` is what survives **across turns**: the running `messages` history (so the
model remembers earlier turns), a `memo` cache (so repeated calls are free), and a
`log` (the observability trace).

- `run_agent` starts a **new** user turn.
- `resume` continues the **same** turn after a confirmation gate has paused it — no new
  user message, the model just sees the earlier "needs confirmation" tool result and
  tries again.
- When the model returns **more than one tool call in the same message**, they're
  independent by construction (that's what we asked for in the system prompt), so
  `_loop` runs them **concurrently** with a thread pool instead of one at a time.

In [ ]:
@dataclass
class Session:
    messages: list = field(default_factory=lambda: [{"role": "system", "content": SYSTEM_PROMPT}])
    memo: dict = field(default_factory=dict)
    log: list = field(default_factory=list)


def _run_one_call(tc, *, session, confirm, tool_impls):
    args = json.loads(tc.function.arguments or "{}")
    impl = (tool_impls or {}).get(tc.function.name)
    try:
        result = execute_tool(tc.function.name, args, log=session.log, impl=impl,
                               memo=session.memo, confirm=confirm)
        return tc.id, {"ok": True, "result": result}
    except ConfirmationRequired as e:
        return tc.id, {"ok": False, "error": "needs_confirmation", "tool": e.tool, "args": e.call_args}
    except Exception as e:
        return tc.id, {"ok": False, "error": str(e)}


def _loop(session, *, max_iterations, confirm, tool_impls):
    iterations = 0
    while iterations < max_iterations:                           # MAX-ITERATION GUARD
        ai_msg = llm_step(session.messages)                       # 1) THINK

        assistant_entry = {"role": "assistant", "content": ai_msg.content}
        if ai_msg.tool_calls:
            assistant_entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in ai_msg.tool_calls
            ]
        session.messages.append(assistant_entry)

        if not ai_msg.tool_calls:                                 # 2a) done -> synthesize
            return ai_msg.content

        calls = ai_msg.tool_calls                                 # 2b) ACT -- fan out if >1
        with ThreadPoolExecutor(max_workers=len(calls)) as ex:
            results = list(ex.map(
                lambda tc: _run_one_call(tc, session=session, confirm=confirm, tool_impls=tool_impls),
                calls))

        for tc_id, r in results:                                  # 3) OBSERVE
            session.messages.append({"role": "tool", "tool_call_id": tc_id, "content": json.dumps(r)})

        blocked = next((r for _, r in results if r.get("error") == "needs_confirmation"), None)
        if blocked:
            return f"Awaiting confirmation to run '{blocked['tool']}' with {blocked['args']}."

        iterations += 1

    return "Stopped: hit the max-iteration guard."


def run_agent(query, session, *, max_iterations=6, confirm=deny_destructive, tool_impls=None):
    """Start a new user turn."""
    session.messages.append({"role": "user", "content": query})
    return _loop(session, max_iterations=max_iterations, confirm=confirm, tool_impls=tool_impls)


def resume(session, *, max_iterations=6, confirm=always_approve, tool_impls=None):
    """Continue the current turn after a confirmation gate paused it."""
    return _loop(session, max_iterations=max_iterations, confirm=confirm, tool_impls=tool_impls)

## 5. Demo — a single-step query

The model calls `get_ticket` once, sees the result, and answers. This is the whole
loop in its simplest form: one THINK → ACT → OBSERVE → THINK(done) cycle.

In [ ]:
s1 = Session()
answer = run_agent("What's the status of ticket TKT-2?", s1)
print(answer)
print()
print_trace(s1.log)

## 6. Demo — multi-step + the confirmation gate

"Close the ticket about login" needs two steps (search, then close), and `close_ticket`
is destructive. The default policy (`deny_destructive`) pauses the agent instead of
just doing it — exactly what you want before an irreversible action.

In [ ]:
s2 = Session()
answer = run_agent("Close the ticket about login", s2, confirm=deny_destructive)
print(answer)

Approve it, and the *same* turn picks up where it left off:

In [ ]:
answer = resume(s2, confirm=always_approve)
print(answer)
print()
print_trace(s2.log)

## 7. Demo — memoization

Called directly here (bypassing the LLM) so the repeated call is byte-for-byte
identical and the cache hit is guaranteed rather than dependent on phrasing. Inside
`run_agent` this same `session.memo` is shared for the whole conversation, so if the
model repeats an identical call later in the turn (or a later turn), it's free too.

In [ ]:
s3 = Session()
execute_tool("search_tickets", {"query": "auth"}, log=s3.log, memo=s3.memo)
execute_tool("search_tickets", {"query": "auth"}, log=s3.log, memo=s3.memo)   # identical -> cache_hit
print_trace(s3.log)

## 8. Demo — retry on a flaky tool

`search_tickets` fails twice with a `TransientError`, then succeeds. The retry loop
inside `execute_tool` makes that invisible to the caller — a `NotFoundError` (permanent)
would NOT be retried; see the fallback demo below for what happens there instead.

In [ ]:
flaky_search = make_flaky(lambda query: store.search(query), fail_times=2)

s4 = Session()
result = execute_tool("search_tickets", {"query": "auth"}, log=s4.log, impl=flaky_search)
print(result)
print()
print_trace(s4.log)

## 9. Demo — fallback when a tool fails permanently

Simulate the primary ticket-lookup path being down (`NotFoundError`, not retryable).
`with_fallback` reroutes to a backup implementation of the same read instead of just
giving up.

In [ ]:
def primary_lookup(**kwargs):
    raise NotFoundError("primary ticket-lookup service is unreachable")

def backup_lookup(ticket_id):
    return store.get(ticket_id)

s5 = Session()
result = with_fallback("get_ticket", "get_ticket_backup", {"ticket_id": "TKT-1"},
                        log=s5.log, impl=primary_lookup, fallback_impl=backup_lookup)
print(result)
print()
print_trace(s5.log)

## 10. Demo — disambiguating two plausible tools

If a query could match more than one tool, don't let anything guess a destructive one
on a tie — resolve it deterministically (or ask the user to confirm).

In [ ]:
choice = disambiguate(["search_tickets", "close_ticket"], "find the ticket about login")
print("chosen tool:", choice)

## 11. Demo — the max-iteration guard

Force the budget down to a single step on a task that genuinely needs two (search,
then close). The guard trips regardless of what the model tries next — it's a hard
stop enforced by the orchestration code, not something left to the model's judgment.

In [ ]:
s6 = Session()
answer = run_agent("Close the ticket about login", s6, max_iterations=1, confirm=always_approve)
print(answer)
print()
print_trace(s6.log)

## 12. Demo — parallel vs. sequential tool calls

Give `get_ticket` a fake 300ms of "network latency" and ask for two tickets at once.
Because the two lookups are independent, the model can request both `get_ticket` calls
in the *same* turn — and `_loop` fans them out across a thread pool instead of running
them one at a time. Watch the `ms` column in the trace: both calls show ~300ms each,
but they overlap, so the *tool-call* portion of the wall time is close to one call's
latency, not the sum of both. (Total wall time below also includes the model's own
round-trip time, which isn't part of this trade-off.)

If the model instead calls them in two separate turns, that's a legitimate model
choice too — nudge the system prompt harder if you want to force batching.

In [ ]:
def slow_get_ticket(ticket_id):
    time.sleep(0.3)
    return store.get(ticket_id)

s7 = Session()
t0 = time.perf_counter()
answer = run_agent("Give me the full details of TKT-1 and TKT-2.", s7,
                    tool_impls={"get_ticket": slow_get_ticket})
elapsed = time.perf_counter() - t0
print(answer)
print(f"\ntotal wall time: {elapsed:.2f}s")
print()
print_trace(s7.log)

## 13. Demo — state across conversation turns

The same `Session` carries the conversation forward in `session.messages`, so a
follow-up like "actually, close it" resolves "it" from the earlier turn — no
re-explaining which ticket. This is the multi-turn memory pointer from the prep notes.

In [ ]:
s8 = Session()
print(run_agent("What's the status of the billing overcharge ticket?", s8))
print()
print(run_agent("Actually, go ahead and close it.", s8, confirm=always_approve))

## 14. Observability — reading a trace

Every entry in `session.log` records what was called, with what args, what happened
(`ok` / `cache_hit` / `retry` / `blocked` / `error`), and latency. That's what you'd
point to for debugging ("why did it close the wrong ticket?"), cost/latency analysis,
or an audit trail of destructive actions. In production this becomes structured spans
— OpenTelemetry or LangSmith — one per tool call, tagged the same way, attached to a
`conversation_id` so you can pull the whole story.

In [ ]:
print_trace(s8.log)

## 15. Design talking points (say these out loud)

**State management across turns**
In-memory `Session` here. In production: a store keyed by `conversation_id`
(Redis/DB) so any worker can resume a conversation, plus a durable checkpointer to
survive restarts and support human-in-the-loop pauses (exactly what the confirmation
gate needs). Cap history size — summarize or window old turns so the prompt doesn't
grow without bound.

**Parallel vs. sequential tool calls**
| | Sequential | Parallel |
|---|---|---|
| Latency | sum of all calls | max of the calls |
| Use when | later calls **depend on** earlier results (search → then close) | calls are **independent** |
| Cost | same total work | same total work, but watch **rate limits** (N calls at once) |

Dependent steps stay sequential — you can't close a ticket before you've found it.
Independent reads go parallel. Parallel calls hit the rate limiter harder, so
coordinate them through the same budget/limiter rather than firing unbounded bursts;
keep a latency budget and return a partial answer rather than blocking on a slow branch.

**Failure policy**
Transient (429 / 503 / timeout) → retry with backoff. Permanent failure with a
fallback available → use the fallback. No fallback → escalate to the user with an
explanation. Never silently swallow a failure inside the loop, and never auto-retry a
*destructive* call without re-checking the confirmation.

**Confirmation gate**
Enforced in the orchestration layer (`execute_tool`), not left to the model's
judgment — a prompt-injected or simply overconfident model should never be able to
close/delete something without the gate seeing it first.

## 16. Recap — what maps to what

| Interview bullet | Where it lives in this notebook |
|---|---|
| Tool registry (name, description, schema, destructive flag) | Section 1 |
| Real LLM routes the query to a tool | Section 3 (`llm_step`, native OpenAI tool calling) |
| ReAct loop: think / act / observe | Section 4 (`_loop`) |
| Max-iteration guard | Section 4 (`while iterations < max_iterations`); Demo in Section 11 |
| Synthesizing a final answer | Section 4, when `ai_msg.tool_calls` is empty |
| Confirmation gate before destructive tools | Section 2 (`execute_tool`); Demo in Section 6 |
| Memoization of repeated calls | Section 2 (`memo` cache); Demo in Section 7 |
| Retry on transient failures | Section 2 (`execute_tool` retry loop); Demo in Section 8 |
| Fallback to a backup tool | Section 2 (`with_fallback`); Demo in Section 9 |
| Disambiguating two plausible tools | Section 2 (`disambiguate`); Demo in Section 10 |
| Parallel vs. sequential tool calls | Section 4 (`ThreadPoolExecutor`); Demo in Section 12 |
| State across conversation turns | `Session.messages` / `Session.memo`; Demo in Section 13 |
| Observability / tool-call logging | `session.log`, `print_trace`; Section 14 |

For the fully modular version of the same ideas — split across `src/` modules, with
an **offline** rule-based brain (no API key needed) *and* the same loop re-expressed
as a **LangGraph** `StateGraph` — see `agent_tool_calling_demo.ipynb` in this folder.